## Ulkeleri askeri guclerine gore gruplara ayirma

1. trillion dolar : bu sadece america icine girer.
2. rusya ve cin bunlar kendi teknolojilerini uretiyorlar ama amerika kadar guclu degiller
3. Turkiye, Brazilya, Almanya, Fransa bunlar teknolojilerinin cogunu disaridan aliyorlar. Ama kendileri de cok az bir sey yapiyorlar.
4. Hic savunma sanayilari yok. or Afganistan denize kiyilari olmadigi icin deniz donanmalari yok.

In [1]:
import pandas as pd

In [2]:
df = pd.read_excel('World military power.xlsx', header=[0, 1])

In [3]:
df.head()

2020 ranking                               Airforce Strength  \
  Military Strength Military Strength Power Index Aircraft Strength   
0       Afghanistan                        1.3444       Afghanistan   
1           Albania                        2.3137           Albania   
2           Algeria                        0.4659           Algeria   
3            Angola                        0.8379            Angola   
4         Argentina                        0.6521         Argentina   

                                                        \
  Aircraft Strength value Fighter/Interceptor Strength   
0                     260                  Afghanistan   
1                      19                      Albania   
2                     551                      Algeria   
3                     295                       Angola   
4                     227                    Argentina   

                                                               \
  Fighter/Interceptor Strength value Attack Aircraft Strength   
0                                  0              Afghanistan   
1                                  0                  Albania   
2                                103                  Algeria   
3                                 72                   Angola   
4                                 24                Argentina   

                                                                    \
  Attack Aircraft Strength value Transport Aircraft Fleet Strength   
0                             25                       Afghanistan   
1                              0                           Albania   
2                             22                           Algeria   
3                             18                            Angola   
4                              7                         Argentina   

                                           ...         Manpower  \
  Transport Aircraft Fleet Strength value  ... Total Population   
0                                      30  ...      Afghanistan   
1                                       0  ...          Albania   
2                                      59  ...          Algeria   
3                                      30  ...           Angola   
4                                       9  ...        Argentina   

                                      Geography                               \
  Total Population value Total Square Land Area Total Square Land Area value   
0            3,49,40,837            Afghanistan                     6,52,230   
1              30,57,220                Albania                        28748   
2            4,16,57,488                Algeria                    23,81,741   
3            3,03,55,880                 Angola                    12,46,700   
4            4,46,94,198              Argentina                    27,80,400   

                                                           \
  Total Coastline Coverage Total Coastline Coverage value   
0              Afghanistan                              0   
1                  Albania                            362   
2                  Algeria                            998   
3                   Angola                           1600   
4                Argentina                           4989   

                                                                               \
  Total Waterway Coverage Total Waterway Coverage value Total Border Coverage   
0             Afghanistan                          1200           Afghanistan   
1                 Albania                            41               Albania   
2                 Algeria                             0               Algeria   
3                  Angola                          1300                Angola   
4               Argentina                         11000             Argentina   

                               
  Total Border Coverage value  
0                      5987.0  
1                       691.0  


In [4]:
df.isnull().sum()

2020 ranking       Military Strength                           0
                   Military Strength Power Index               0
Airforce Strength  Aircraft Strength                           0
                   Aircraft Strength value                     0
                   Fighter/Interceptor Strength                0
                   Fighter/Interceptor Strength value          0
                   Attack Aircraft Strength                    0
                   Attack Aircraft Strength value              0
                   Transport Aircraft Fleet Strength           0
                   Transport Aircraft Fleet Strength value     0
                   Trainer Aircraft Fleet                      0
                   Trainer Aircraft Fleet value                0
                   Helicopter Fleet Strength                   0
                   Helicopter Fleet Strength value             0
                   Attack Helicopter Fleet Strength            0
                   Attack

In [5]:
#pip install scikit-learn

In [12]:

import numpy as np
from sklearn.preprocessing import StandardScaler

# Pandas'ın "Unnamed" bıraktığı boş ana başlıkları otomatik dolduruyoruz
yeni_ana_basliklar = []
guncel_baslik = ""
for ana, alt in df.columns:
    if "Unnamed" not in ana:
        guncel_baslik = ana
    yeni_ana_basliklar.append((guncel_baslik, alt))
df.columns = pd.MultiIndex.from_tuples(yeni_ana_basliklar)

# 2. Üzerinde birleştirme yapacağımız asıl askeri/ekonomik ana başlıkları seçiyoruz
# '2020 ranking' başlığını bu döngünün dışında tutuyoruz ki yapısı bozulmasın
ana_basliklar = list(dict.fromkeys([col[0] for col in df.columns if "ranking" not in col[0].lower()]))

# 3. EN BAŞTAKİ DEĞERLERİ KORUMA: Sıralama sütunundan Ülke ismi ve Power Index'i alıyoruz
en_bastan_ulkeler = df.iloc[:, 0]
power_index_serisi = df.iloc[:, 1]

# Power Index'teki sayısal formatı temizleyelim ve sayıya çevirelim
power_index_serisi = power_index_serisi.astype(str).str.replace(',', '')
power_index_serisi = pd.to_numeric(power_index_serisi, errors='coerce')

# Yeni tablomuzun temelini oluşturuyoruz (Sıralama verileri korundu)
yeni_veri = {
    'Country': en_bastan_ulkeler,
    'Military Strength Power Index': power_index_serisi
}

# 4. Diğer tüm alt sütunları ÖNCE ÖLÇEKLENDİRİP SONRA tek sütuna indirgeme döngüsü
for ana in ana_basliklar:
    alt_grup = df[ana]
    
    # Sadece sayısal değer içeren alt sütunları yakala
    sayisal_alt_sutunlar = [
        col for col in alt_grup.columns 
        if any(anahtar in col.lower() for anahtar in ['value', 'budget', 'total', 'production', 'consumption', 'reserves', 'population', 'coverage'])
    ]
    
    if len(sayisal_alt_sutunlar) == 0:
        continue
        
    # Alt sütunlardaki verileri temizle ve sayıya çevir
    temiz_alt_grup = alt_grup[sayisal_alt_sutunlar].copy()
    for col in temiz_alt_grup.columns:
        temiz_alt_grup[col] = temiz_alt_grup[col].astype(str).str.replace(',', '')
        temiz_alt_grup[col] = pd.to_numeric(temiz_alt_grup[col], errors='coerce')
        # Boş (NaN) değerler varsa, her alt sütunu kendi medyanı ile dolduralım ki scale düzgün çalışsın
        temiz_alt_grup[col] = temiz_alt_grup[col].fillna(temiz_alt_grup[col].median())
    
    # --- YENİ MANTIK: ÖNCE ÖLÇEKLENDİRME ---
    # Her alt sütunu kendi içinde StandardScaler ile ölçeklendiriyoruz
    scaler = StandardScaler()
    olcekli_alt_grup = pd.DataFrame(
        scaler.fit_transform(temiz_alt_grup), 
        columns=temiz_alt_grup.columns, 
        index=temiz_alt_grup.index
    )
    
    # --- SONRA BİRLEŞTİRME ---
    # Ölçeklenmiş (artık aynı elma/armut kıyaslamasına gelen) alt sütunların satır bazında ortalamasını alıyoruz
    tek_sutun_ortalama = olcekli_alt_grup.mean(axis=1)

    scaler_nihai = StandardScaler()
    nihai_olcekli_sutun = scaler_nihai.fit_transform(tek_sutun_ortalama.values.reshape(-1, 1)).flatten()
    
    # Oluşan bu nihai skoru, ana başlığın adıyla sözlüğümüze ekliyoruz
    yeni_veri[ana] = nihai_olcekli_sutun

# 5. Sözlüğü düz ve temiz bir DataFrame'e dönüştür
final_df = pd.DataFrame(yeni_veri)


/home/zaid/Desktop/Yapay-Zeka-Kursu/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:1211: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/home/zaid/Desktop/Yapay-Zeka-Kursu/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:1216: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/home/zaid/Desktop/Yapay-Zeka-Kursu/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:1240: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/home/zaid/Desktop/Yapay-Zeka-Kursu/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:1211: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/home/zaid/Desktop/Yapay-Zeka-Kursu/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:1216: RuntimeWarning: invalid value encountered in divide
  T = new_sum / 

In [13]:
final_df.head()

,Country,Military Strength Power Index,Airforce Strength,Land Strength,Navy Strength,Finances,Logistics,Natural resources,Manpower,Geography
0,Afghanistan,1.3444,-0.150608,-0.326811,-0.443378,-0.161792,-0.204411,-0.390521,-0.121546,-0.031768
1,Albania,2.3137,-0.315349,-0.421517,-0.373848,-0.251213,-0.236680,-0.377968,-0.294630,-0.599058
2,Algeria,0.4659,0.135463,0.296032,0.255576,-0.153053,-0.110084,0.108442,-0.062990,0.274568
3,Angola,0.8379,-0.076414,-0.203213,-0.339083,-0.189535,-0.094363,0.113439,-0.183519,0.028668
4,Argentina,0.6521,-0.186578,-0.271830,-0.202134,-0.162357,0.701623,-0.123010,-0.057093,1.068812
